# Unity Catalog Admin Setup

This notebook handles admin-level UC operations:
1. Register S3 credential & external location
2. Grant permissions to users (metastore, external location)
3. Verify setup

After running this, users can create their own catalogs (with managed storage), schemas, and tables.

**Requires:** UC admin token (from pod) and port-forward to UC (8080)

## Step 0: Configuration

In [ ]:
import os
import requests
import json

UC_ENDPOINT = os.environ.get("UNITY_CATALOG_ENDPOINT", "http://server.unity-catalog.svc.cluster.local:8080")
print("UC_ENDPOINT:", UC_ENDPOINT)

# Paste UC admin token here
# Get it via: kubectl exec -n unity-catalog $(kubectl get pod -n unity-catalog -l app=unity-catalog -o jsonpath='{.items[0].metadata.name}') -- cat /home/unitycatalog/etc/conf/token.txt
UC_ADMIN_TOKEN = ""

headers = {"Authorization": f"Bearer {UC_ADMIN_TOKEN}"}

## Step 1: Create S3 Credential

Register an AWS IAM role as a storage credential in UC.
This is required before creating external locations.

In [ ]:
CREDENTIAL_NAME = "tuantm-s3-cred"
ROLE_ARN = "arn:aws:iam::484846800028:role/tuantm-uc-role"

resp = requests.post(
    f"{UC_ENDPOINT}/api/2.1/unity-catalog/credentials",
    headers=headers,
    json={
        "name": CREDENTIAL_NAME,
        "purpose": "STORAGE",
        "aws_iam_role": {"role_arn": ROLE_ARN},
    },
)
if resp.status_code == 200:
    print(f"Created credential: {CREDENTIAL_NAME}")
elif resp.status_code == 409:
    print(f"Credential already exists: {CREDENTIAL_NAME}")
else:
    print(f"Create credential response: {resp.status_code} - {resp.text}")

In [ ]:
# Verify credentials
resp = requests.get(f"{UC_ENDPOINT}/api/2.1/unity-catalog/credentials", headers=headers)
print(json.dumps(resp.json(), indent=2))

## Step 2: Create External Location

Map an S3 path to the credential. Users need this to exist before they can
create catalogs with `storage_root` for managed tables.

In [ ]:
EXTERNAL_LOCATION_NAME = "tuantm-s3"
S3_URL = "s3://tuantm-data-platform"

resp = requests.post(
    f"{UC_ENDPOINT}/api/2.1/unity-catalog/external-locations",
    headers=headers,
    json={
        "name": EXTERNAL_LOCATION_NAME,
        "url": S3_URL,
        "credential_name": CREDENTIAL_NAME,
    },
)
if resp.status_code == 200:
    print(f"Created external location: {EXTERNAL_LOCATION_NAME} -> {S3_URL}")
elif resp.status_code == 409:
    print(f"External location already exists: {EXTERNAL_LOCATION_NAME}")
else:
    print(f"Create external location response: {resp.status_code} - {resp.text}")

In [ ]:
# Verify external locations
resp = requests.get(f"{UC_ENDPOINT}/api/2.1/unity-catalog/external-locations", headers=headers)
print(json.dumps(resp.json(), indent=2))

## Step 3: Register User in UC

UC requires users to exist in its database before Keycloak token exchange works.
The email must match the email in the Keycloak user.

In [ ]:
USER_EMAIL = "snaketuan@gmail.com"
USER_DISPLAY_NAME = "Tuan"

resp = requests.post(
    f"{UC_ENDPOINT}/api/1.0/unity-control/scim2/Users",
    headers=headers,
    json={
        "displayName": USER_DISPLAY_NAME,
        "emails": [{"value": USER_EMAIL, "primary": True}],
    },
)
if resp.status_code == 200:
    print(f"Created user: {USER_EMAIL}")
    print(json.dumps(resp.json(), indent=2))
elif resp.status_code == 409:
    print(f"User already exists: {USER_EMAIL}")
else:
    print(f"Create user response: {resp.status_code} - {resp.text}")

## Step 3: Grant Permissions to User

Grant the user:
- `CREATE CATALOG` on metastore — so they can create their own catalogs
- `CREATE MANAGED STORAGE` on external location — so they can set `storage_root` on catalogs (for managed tables)
- `CREATE EXTERNAL TABLE` on external location — so they can create external tables under this S3 path

Since the user creates their own catalog/schema, they become **OWNER** and automatically
get `USE CATALOG`, `USE SCHEMA`, `CREATE SCHEMA`, `CREATE TABLE`, `SELECT`, `MODIFY` etc.

In [ ]:
USER_EMAIL = "snaketuan@gmail.com"
EXTERNAL_LOCATION_NAME = "tuantm-s3"

# Grant CREATE CATALOG on metastore
resp = requests.patch(
    f"{UC_ENDPOINT}/api/2.1/unity-catalog/permissions/metastore/metastore",
    headers=headers,
    json={"changes": [{"principal": USER_EMAIL, "add": ["CREATE CATALOG"]}]},
)
print(f"Metastore permissions: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

# Grant CREATE MANAGED STORAGE + CREATE EXTERNAL TABLE on external location
resp = requests.patch(
    f"{UC_ENDPOINT}/api/2.1/unity-catalog/permissions/external_location/{EXTERNAL_LOCATION_NAME}",
    headers=headers,
    json={"changes": [{"principal": USER_EMAIL, "add": ["CREATE MANAGED STORAGE", "CREATE EXTERNAL TABLE"]}]},
)
print(f"External location permissions: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Step 4: Verify Setup

In [ ]:
# List users
resp = requests.get(f"{UC_ENDPOINT}/api/1.0/unity-control/scim2/Users", headers=headers)
print("=== Users ===")
print(json.dumps(resp.json(), indent=2))

In [ ]:
USER_EMAIL = "snaketuan@gmail.com"
EXTERNAL_LOCATION_NAME = "tuantm-s3"

# Check permissions
for securable, name in [("metastore", "metastore"), ("external_location", EXTERNAL_LOCATION_NAME)]:
    resp = requests.get(
        f"{UC_ENDPOINT}/api/2.1/unity-catalog/permissions/{securable}/{name}",
        headers=headers,
        params={"principal": USER_EMAIL},
    )
    privileges = resp.json().get("privilege_assignments", [])
    privs = privileges[0]["privileges"] if privileges else []
    print(f"{securable}/{name}: {privs}")

In [ ]:
# List credentials and external locations
resp = requests.get(f"{UC_ENDPOINT}/api/2.1/unity-catalog/credentials", headers=headers)
print("=== Credentials ===")
print(json.dumps(resp.json(), indent=2))

resp = requests.get(f"{UC_ENDPOINT}/api/2.1/unity-catalog/external-locations", headers=headers)
print("\n=== External Locations ===")
print(json.dumps(resp.json(), indent=2))

In [ ]:
# List catalogs
resp = requests.get(f"{UC_ENDPOINT}/api/2.1/unity-catalog/catalogs", headers=headers)
print("=== Catalogs ===")
for c in resp.json().get("catalogs", []):
    print(f"  {c['name']} | storage_root: {c.get('storage_root', 'N/A')} | storage_location: {c.get('storage_location', 'N/A')}")